# Apache Iceberg com PySpark

Aqui a gente demonstra as operações principais do Apache Iceberg com PySpark.
O dataset é o mesmo do Delta Lake — vendas de um e-commerce fictício — pra facilitar a comparação.

Operações cobertas: criação da tabela via catálogo, INSERT, UPDATE, DELETE e consulta de snapshots.

> **Obs:** na primeira execução o Spark vai baixar o JAR do Iceberg via Maven, pode demorar um pouco.

## Configurando o SparkSession com Iceberg

O Iceberg precisa do JAR de runtime. A forma mais simples é passar via `PYSPARK_SUBMIT_ARGS` antes de criar a sessão.

In [ ]:
import os

# define o pacote iceberg antes de iniciar o spark
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Apache Iceberg Demo") \
    .master("local[*]") \
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    ) \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "./iceberg-warehouse") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark versao: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

## Criando o namespace e a tabela Iceberg

In [ ]:
# cria o namespace (equivalente ao database)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

# dropa a tabela caso ja exista (pra poder reexecutar o notebook)
spark.sql("DROP TABLE IF EXISTS local.db.vendas")

spark.sql("""
    CREATE TABLE local.db.vendas (
        id         INT,
        produto    STRING,
        categoria  STRING,
        quantidade INT,
        preco      DOUBLE,
        data_venda STRING,
        vendedor   STRING,
        status     STRING
    )
    USING iceberg
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.parquet.compression-codec' = 'snappy'
    )
""")

print("tabela criada:")
spark.sql("DESCRIBE TABLE local.db.vendas").show(truncate=False)

## Definindo o schema e os dados

In [ ]:
schema = StructType([
    StructField("id",         IntegerType(), False),
    StructField("produto",    StringType(),  True),
    StructField("categoria",  StringType(),  True),
    StructField("quantidade", IntegerType(), True),
    StructField("preco",      DoubleType(),  True),
    StructField("data_venda", StringType(),  True),
    StructField("vendedor",   StringType(),  True),
    StructField("status",     StringType(),  True),
])

dados_iniciais = [
    (1,  "Notebook Dell",     "Eletronicos", 2, 3500.00, "2024-01-15", "Ana Silva",    "entregue"),
    (2,  "Camiseta Polo",     "Roupas",      5,   89.90, "2024-01-20", "Bruno Costa",  "entregue"),
    (3,  "Tenis Running",     "Calcados",    1,  450.00, "2024-02-01", "Carlos Lima",  "pago"),
    (4,  "Mouse Gamer",       "Eletronicos", 3,  250.00, "2024-02-10", "Diana Ramos",  "entregue"),
    (5,  "Calca Jeans",       "Roupas",      2,  199.90, "2024-02-14", "Eduardo Melo", "entregue"),
    (6,  "Smartphone X",      "Eletronicos", 1, 2800.00, "2024-02-20", "Ana Silva",    "pago"),
    (7,  "Mochila Viagem",    "Acessorios",  1,  350.00, "2024-03-01", "Bruno Costa",  "pendente"),
    (8,  "Teclado Mecanico",  "Eletronicos", 1,  450.00, "2024-03-05", "Carlos Lima",  "pago"),
    (9,  "Vestido Floral",    "Roupas",      3,  159.90, "2024-03-10", "Diana Ramos",  "entregue"),
    (10, "Monitor 27pol",     "Eletronicos", 1, 1800.00, "2024-03-15", "Eduardo Melo", "pendente"),
    (11, "Sandalia Couro",    "Calcados",    2,  280.00, "2024-03-18", "Ana Silva",    "entregue"),
    (12, "Camera DSLR",       "Eletronicos", 1, 4200.00, "2024-03-20", "Bruno Costa",  "pago"),
    (13, "Jaqueta Denim",     "Roupas",      1,  320.00, "2024-03-22", "Carlos Lima",  "cancelado"),
    (14, "Headset USB",       "Eletronicos", 2,  180.00, "2024-03-25", "Diana Ramos",  "cancelado"),
    (15, "Relogio Analogico", "Acessorios",  1,  520.00, "2024-03-28", "Eduardo Melo", "pendente"),
]

print(f"Total de registros: {len(dados_iniciais)}")

## INSERT — carga inicial

In [ ]:
df_inicial = spark.createDataFrame(dados_iniciais, schema)

df_inicial.writeTo("local.db.vendas").append()

total = spark.sql("SELECT COUNT(*) AS total FROM local.db.vendas").collect()[0][0]
print(f"Registros inseridos: {total}")

spark.sql("SELECT * FROM local.db.vendas").show(truncate=False)

### INSERT — novos pedidos via SQL

In [ ]:
spark.sql("""
    INSERT INTO local.db.vendas VALUES
    (16, 'Fone Bluetooth', 'Eletronicos', 1, 299.90, '2024-04-01', 'Ana Silva',   'pendente'),
    (17, 'Bolsa de Couro', 'Acessorios',  1, 480.00, '2024-04-02', 'Bruno Costa', 'pendente'),
    (18, 'Tenis Social',   'Calcados',    1, 390.00, '2024-04-03', 'Carlos Lima', 'pendente')
""")

total = spark.sql("SELECT COUNT(*) AS total FROM local.db.vendas").collect()[0][0]
print(f"Total agora: {total}")

spark.sql("SELECT * FROM local.db.vendas WHERE id >= 16").show(truncate=False)

## UPDATE — atualizando registros

In [ ]:
print("pedidos pendentes antes do update:")
spark.sql("""
    SELECT id, produto, vendedor, status FROM local.db.vendas WHERE status = 'pendente'
""").show(truncate=False)

# atualiza pendente -> pago
spark.sql("""
    UPDATE local.db.vendas SET status = 'pago' WHERE status = 'pendente'
""")

print("depois do update:")
spark.sql("""
    SELECT id, produto, vendedor, status FROM local.db.vendas WHERE id IN (7,10,15,16,17,18)
""").show(truncate=False)

In [ ]:
# segundo update: pagos mais antigos passam pra entregue
spark.sql("""
    UPDATE local.db.vendas
    SET status = 'entregue'
    WHERE status = 'pago' AND data_venda < '2024-03-01'
""")

print("contagem por status:")
spark.sql("""
    SELECT status, COUNT(*) AS total FROM local.db.vendas GROUP BY status ORDER BY status
""").show()

## DELETE — removendo registros

In [ ]:
print("registros cancelados que serao removidos:")
spark.sql("SELECT * FROM local.db.vendas WHERE status = 'cancelado'").show(truncate=False)

spark.sql("DELETE FROM local.db.vendas WHERE status = 'cancelado'")

total_pos = spark.sql("SELECT COUNT(*) FROM local.db.vendas").collect()[0][0]
print(f"Registros restantes: {total_pos}")

spark.sql("""
    SELECT status, COUNT(*) AS total FROM local.db.vendas GROUP BY status ORDER BY status
""").show()

## Snapshots e Time Travel

Cada operação no Iceberg gera um snapshot. Diferente do Delta Lake (que usa um transaction log em JSON), o Iceberg usa arquivos de manifesto para rastrear o estado da tabela.

In [ ]:
print("snapshots gerados:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation FROM local.db.vendas.snapshots
""").show(truncate=False)

In [ ]:
# pega o id do primeiro snapshot (insercao inicial)
primeiro_snapshot = spark.sql("""
    SELECT snapshot_id FROM local.db.vendas.snapshots ORDER BY committed_at ASC
""").collect()[0][0]

print(f"lendo o snapshot original (id: {primeiro_snapshot}):")
df_original = spark.read \
    .option("snapshot-id", primeiro_snapshot) \
    .table("local.db.vendas")

print(f"registros no snapshot original: {df_original.count()}")

cancelados_original = df_original.filter(col("status") == "cancelado").count()
cancelados_atual    = spark.sql("SELECT COUNT(*) FROM local.db.vendas WHERE status = 'cancelado'").collect()[0][0]
print(f"cancelados no snapshot original: {cancelados_original}  |  na versao atual: {cancelados_atual}")

## Estado final e estatísticas

In [ ]:
print("tabela final:")
spark.sql("SELECT * FROM local.db.vendas ORDER BY id").show(truncate=False)

print("estatisticas por categoria:")
spark.sql("""
    SELECT
        categoria,
        COUNT(id)                         AS total_pedidos,
        SUM(quantidade)                   AS total_itens,
        ROUND(SUM(preco * quantidade), 2) AS receita_total,
        ROUND(AVG(preco), 2)              AS preco_medio
    FROM local.db.vendas
    GROUP BY categoria
    ORDER BY receita_total DESC
""").show(truncate=False)

spark.stop()